# Using the Analytics Engine (AE) to reproduce annual consumption model
This notebook covers:
1. selecting data to work with
2. retrieving a dataset from the catalog
3. a simple plot to preview the data
4. how to export that data

To execute a given 'cell' of this notebook, place the cursor in the cell and press the 'play' icon, or simply press shift+enter together. Some cells will take longer to run, and you will see a [$\ast$] to the left of the cell while AE is still working.

## Step 0: Setup
This cell imports the python library [climakitae](https://github.com/cal-adapt/climakitae), our AE toolkit for climate data analysis, and any other specialized libraries needed for a given notebook.

In [ ]:
import numpy as np
import xarray as xr
import hvplot.xarray
import panel as pn
pn.extension()

In [ ]:
import climakitae as ck

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

## Step 1: Select data
Now we can call 'select' to display an interface from which to select the data to examine. Execute the cell, and read on for more explanation.

Currently, you can select from [dynamically-downscaled](https://dept.atmos.ucla.edu/alexhall/downscaling-cmip6) data produced at hourly intervals. If you select 'daily' or 'monthly' for 'Timescale', you will receive an average of the hourly data. The spatial resolution options, on the other hand, are each the output of a different simulation, nesting to higher resolution over smaller areas.

Future projections are available for a [greenhouse gas emission scenario (Shared Socioeconomic Pathway, or SSP)](https://climatescenarios.org/primer/socioeconomic-development) through 2100 for SSP 3-7.0 for 5 General Circulation Models (GCMs).

At 45 and 9km, more GCMs are to come, and one GCM was also downscaled for a higher and lower SSP. (Later, statistical downscaling will also be available at 3km for more GCMs.)

“Historical Climate” includes data from 1980-2014 simulated from the same GCMs used to produce the SSPs. They can be appended to a SSP time series using the option “Append Historical Climate.” Because this historical data is obtained through simulations, it represents average weather during the historical period and is not meant to capture historical timeseries as they occurred. 

“Historical Reconstruction” provides a reference downscaled [reanalysis](https://www.ecmwf.int/en/about/media-centre/focus/2020/fact-sheet-reanalysis) dataset based on atmospheric models fit to satellite and station observations, and as a result will reflect observed historical time-evolution of the weather.

To learn more about the data available on the Analytics Engine, [see our data catalog](https://analytics.cal-adapt.org/data/). 

In [ ]:
app.select()

Nothing is required to enter these selections, besides moving on to Step 2.

However, if you want to preview what has been selected, you can type "app.selections" alone in a new cell, and "app.location". These store your selections behind-the-scenes.

($+$ will create a new cell, following the currently selected) 

## Step 2: Compute the temperature difference between future climate and historical climate for each month in the year

### 2a) Retrieve the data 
First, modify the app.selections object with the shared data parameters for the analysis in this notebook. 

In [ ]:
%%capture

app.location.area_subset = 'states'
app.location.cached_area = 'CA' # Just get data for the state of California

app.selections.scenario = ['SSP 3-7.0 -- Business as Usual']
app.selections.timescale = 'monthly'
app.selections.resolution = '9 km'
app.selections.units = 'degF'
app.selections.variable = 'Air Temperature at 2m'

Next, modify the temporal range to retrieve historical and future data from AE's data catalog. The data is loaded into an xarray [DataArray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.html) object. 

In [ ]:
app.selections.append_historical = True
app.selections.time_slice = (1981,2020) 

data_hist = app.retrieve()

In [ ]:
app.selections.append_historical = False
app.selections.time_slice = (2061,2100) 

data_fut = app.retrieve()

### 2b) Find the temperature difference between future and historical monthly climate

First, group each dataset (historical and future) by months of the year, then calculate the mean value for each month across the entire time range for each dataset. For example, for the historical data, we will compute the mean December air temperature for all Decembers from 1981-2020. 

In [ ]:
mean_fut_by_month = data_fut.squeeze().groupby('time.month').mean('time') 
mean_hist_by_month = data_hist.squeeze().groupby('time.month').mean('time')

Finally, take the difference between the projected future mean temperature and the historical mean temperature for each month in the year, resulting in the data product `delta_t`.

In [ ]:
delta_t = mean_fut_by_month - mean_hist_by_month
delta_t.name = "2m Temp Difference (future - historical)" 

### 2c) Preview the retrieved, aggregated dataset

In [ ]:
display(delta_t)

### 2d) Load the data into memory
This step may take a few minutes to compute, because the data is only loaded "lazily" until you output it (in visualize or export). This allows the previous steps to run faster.

In [ ]:
%%time
delta_t = app.load(delta_t)

### 2e) Visualize data
Our helper function `app.view()` will allow you to see what the final data product looks like.

In [ ]:
app.view(delta_t)

## Step 3: Compute heating degree days and cooling degree days
Here, a heating degree day (HDD) is calculated by computing how many degrees Farenheit **colder** the daily temperature is from a standard temperature of 65 degrees Farenheit. A cooling degree day (CDD) is calulcated by computing how many degrees **warmer** the daily temperature is from the same standard temperature. **We would like to confirm that these are the same parameters used by the Demand Forecasting Unit to compute HDD and CDD.**

### 3a) Read in the 2m Air Temperature data
Here, we read in daily 2m Air Temperature data, which is pre-calculated from average hourly data and stored as a separate data variable in our data catalog. For this notebook, we chose to work with daily data because it's computationally lighter, but we could also use the hourly data here instead, and manually compute a mean. 

In [ ]:
app.selections.time_slice = (2020, 2100)
app.selections.timescale = 'daily'
app.selections.append_historical = False
app.selections.units = "degF"

t2 = app.retrieve()

### 3b) Compute HDD and CDD
HDD = 65 - temperature<br>
CDD = (-1)\*(65 - temperature)<br><br>
For HDD, we can just subtract the 2m temperature from 65 degrees Farenheight, then set any negative to 0. For CDD, we will do the same, but will then multiply by -1 to turn negative values to positive, then set negative values to 0. We need to multiply by -1 for CDD to avoid having all negative values; for example, on a day of 80F, CDD = 65 - 80 = -15, but the CDD value is +15. Multiplying -15 by -1 will give us the true value of 15. 

In [ ]:
# Subtract t2 from the standard reference temperature 
deg_less_than_standard = 65 - t2

# Compute HDD: Find positive difference (i.e. days < 65 degF) 
hdd = deg_less_than_standard.where(deg_less_than_standard > 0, 0) # Replace negative values with 0
hdd.name = "Heating Degree Days" 

# Compute CDD: Find negative difference (i.e. days > 65 degF)
cdd = (-1)*deg_less_than_standard.where(deg_less_than_standard < 0, 0) # Replace positive values with 0
cdd.name = "Cooling Degree Days" 

### 3c) Aggregate annually to find HDD and CDD per year

In [ ]:
hdd_annual = hdd.squeeze().groupby('time.year').sum(['x','y','time']) # Aggregate annually 
hdd_annual.name = "Annual Heating Degree Days (CDD)" # Rename dataset 

cdd_annual = cdd.squeeze().groupby('time.year').sum(['x','y','time'])
cdd_annual.name = "Annual Cooling Degree Days (HDD)" 

Next, compute the mean trend line across all simulations. First, we will compute then simulation mean, then perform a linear regression using the mean data to find the polynomial coefficients. 

### 3d) Load the data into memory 
This may take some time because we are using high resolution data and have performed a lot of computations that will need to be completed to get the final data product. 

In [ ]:
hdd_annual = app.load(hdd_annual) 
cdd_annual = app.load(cdd_annual) 

### 3d) Compute a trendline using the mean of all simulations

First, we'll compute the mean across all simulations. Then, we'll find the coefficients for a first degree (linear) polynomial using [numpy's `polyfit` function](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html). The returned coefficients (**m** and **b** in the code below) will allow us to compute the trendline using the linear polynomial y = mx + b, where **y** is the trendline and **x** is the years. 

In [ ]:
def trendline(data): 
    years = data.year # These are the x values 
    data_mean = data.mean(dim = "simulation") # Compute mean across all simulations 
    m, b = np.polyfit(
        x = years.values, 
        y = data_mean.values, 
        deg = 1
    ) 
    trendline = m*years + b # y = mx + b 
    trendline.name = "trendline" 
    return trendline

In [ ]:
hdd_trendline = trendline(hdd_annual) 
cdd_trendline = trendline(cdd_annual) 

### 3e) Visualize the results
Using the python package *hvplot*, we can easily make a line plot of the annual aggregated data. To do this, we'll plot the annual HDD, then add the trendline on top. The code to generate the plot is contained in a function `plot_data` defined below. 

In [ ]:
def plot_data(annual_data, trendline, title = "title"): 
    return annual_data.hvplot.line(
        x = "year", by = "simulation", 
        width = 800, height = 350,
        title = title,
        yformatter = '%.0f' # Remove scientific notation
    ) * trendline.hvplot.line(  # Add trendline
        x = "year", 
        color = "black", line_dash = 'dashed', 
        label = "trendline"
    ) 

In [ ]:
plot_data(
    annual_data = hdd_annual, 
    trendline = hdd_trendline, 
    title = "Annual Aggregate Heating Degree Days in California"
)

In [ ]:
plot_data(
    annual_data = cdd_annual, 
    trendline = cdd_trendline, 
    title = "Annual Aggregate Cooling Degree Days in California"
)

## Step 5: Export data

To export, first pick a format from the dropdown menu.
- We recommend NetCDF, which will work with any number of variables and dimensions in your dataset
- CSV and GeoTIFF can only be used for data arrays with one variable
- CSV works best for up to 2-dimensional data (e.g., lon x lat), and will be compressed and exported with a separate metadata file
- GeoTIFF can accept 3 dimensions total: 
    - X and Y dimensions are required
    - The third dimension is flexible and will be a "band" in the file: time, simulation, or scenario could go here
    - Metadata will be accessible as "tags" in the .tif

In [ ]:
app.export_as()

Next, write in the object you wish to export and your desired filename (in single or double quotation marks).

In [ ]:
app.export_dataset(delta_t, 'my_filename')